# Fair-Seldonian on real data: UCI Adult income

This notebook applies the Quasi-Seldonian Algorithm (QSA) to the UCI Adult
income dataset and shows both sides of the Seldonian guarantee on real data:

* an ordinary logistic regression reaches good accuracy but has a measurable
  true-positive-rate (TPR) gap between demographic groups, and
* QSA, asked to certify that gap is bounded, returns **No Solution Found**
  rather than shipping the biased model.

**Task framing**

| symbol | meaning |
|--------|---------|
| `Y` (label) | income > 50K |
| `T` (sensitive) | sex (1 = Male, 0 = Female) |
| `X` (features) | standardized numeric columns, with `T` appended as the final column (library convention) |

> Requires network access on the first run to download the dataset (cached afterwards).

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

from fair_seldonian.algorithms import QSA
from fair_seldonian.models import predict, simple_logistic

## 1. Load and frame the data

We download Adult, derive the binary label and sensitive attribute, standardize
the numeric features, and append the sensitive attribute as the final feature
column. Then we take a deterministic subsample and train/test split.

In [2]:
frame = fetch_openml("adult", version=2, as_frame=True, parser="auto").frame.dropna()

T = (frame["sex"].astype(str) == "Male").astype(int).to_numpy()
Y = frame["class"].astype(str).str.contains(">50K").astype(int).to_numpy()

numeric = frame.select_dtypes("number")
standardized = (numeric - numeric.mean()) / numeric.std()
X = np.column_stack([standardized.to_numpy(), T]).astype(float)

# deterministic subsample + split for a fast, reproducible demo
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))[:8000]
X, Y, T = X[idx], Y[idx], T[idx]
cut = int(0.7 * len(X))
X_tr, Y_tr, T_tr = X[:cut], Y[:cut], T[:cut]
X_te, Y_te, T_te = X[cut:], Y[cut:], T[cut:]

print(
    f"Adult: {len(X)} examples, positive rate {Y.mean():.3f}, male share {T.mean():.3f}"
)

Adult: 8000 examples, positive rate 0.241, male share 0.678


## 2. Unconstrained baseline

A plain logistic regression. We measure overall accuracy and the true-positive
rate within each group; the gap between them is exactly the disparity the
fairness constraint targets.

In [3]:
def true_positive_rate(pred, Y, mask):
    # P(pred = 1 | Y = 1) within the group selected by mask
    positives = mask & (Y == 1)
    return float(pred[positives].mean()) if positives.any() else float("nan")


theta, theta1 = simple_logistic(X_tr, Y_tr)
pred = (predict(theta, theta1, X_te).detach().numpy() >= 0.5).astype(int)

acc = float((pred == Y_te).mean())
tpr_male = true_positive_rate(pred, Y_te, T_te == 1)
tpr_female = true_positive_rate(pred, Y_te, T_te == 0)
gap = abs(tpr_male - tpr_female)

print(f"accuracy      : {acc:.3f}")
print(f"TPR (male)    : {tpr_male:.3f}")
print(f"TPR (female)  : {tpr_female:.3f}")
print(f"TPR gap |M-F| : {gap:.3f}   <- the bias QSA guards against")

accuracy      : 0.814
TPR (male)    : 0.448
TPR (female)  : 0.271
TPR gap |M-F| : 0.177   <- the bias QSA guards against


## 3. The Seldonian guarantee

Now we ask QSA to return a model only if it can certify the equalized-opportunity
constraint holds with high probability. On this data it declines.

In [4]:
_, _, passed = QSA(X_tr, Y_tr, T_tr, "opt", None, None)

if passed:
    print("certified fair")
else:
    print("No Solution Found - QSA will not certify a model on this data,")
    print("rather than return one with the disparity shown above.")

No Solution Found - QSA will not certify a model on this data,
rather than return one with the disparity shown above.


## Takeaway

The unconstrained model is accurate but encodes a real group disparity. QSA trades
coverage for safety: on data where it cannot *prove* the constraint holds, it
returns no model at all - never an unsafe one. See
[`quickstart.ipynb`](quickstart.ipynb) for cases where QSA *does* certify.